In [1]:
# Task 1 — Load and Inspect Data

import pandas as pd

csv_data = """id,name,age,experience,city,expected_salary
1,Ali,22,1,Islamabad,40000
2,Sara,,3,Lahore,60000
3,Ahmed,19,,Karachi,?
4,Bilal,30,7,Peshawar,90000
5,Hira,28,5,isl,65000
6,,26,4,Lahore,70000
"""

with open("applicants.csv", "w") as f:
    f.write(csv_data)

json_data = """[
  {"id": 1, "skills": ["python", "sql"], "education": "bachelors"},
  {"id": 2, "skills": ["excel", "communication"], "education": "masters"},
  {"id": 3, "skills": null, "education": "bachelors"},
  {"id": 4, "skills": ["management"], "education": "phd"},
  {"id": 5, "skills": ["python", "ml"], "education": null}
]"""

with open("applicants_extra.json", "w") as f:
    f.write(json_data)

df1 = pd.read_csv("applicants.csv")
df2 = pd.read_json("applicants_extra.json")

print("=== applicants.csv ===")
print("\nFirst 5 rows:")
print(df1.head())

print("\nData Types:")
print(df1.dtypes)

print("\nMissing Values:")
print(df1.isnull().sum())

print("\nShape:")
print(df1.shape)

print("\n\n=== applicants_extra.json ===")
print("\nFirst 5 rows:")
print(df2.head())

print("\nData Types:")
print(df2.dtypes)

print("\nMissing Values:")
print(df2.isnull().sum())

print("\nShape:")
print(df2.shape)

=== applicants.csv ===

First 5 rows:
   id   name   age  experience       city expected_salary
0   1    Ali  22.0         1.0  Islamabad           40000
1   2   Sara   NaN         3.0     Lahore           60000
2   3  Ahmed  19.0         NaN    Karachi               ?
3   4  Bilal  30.0         7.0   Peshawar           90000
4   5   Hira  28.0         5.0        isl           65000

Data Types:
id                   int64
name                object
age                float64
experience         float64
city                object
expected_salary     object
dtype: object

Missing Values:
id                 0
name               1
age                1
experience         1
city               0
expected_salary    0
dtype: int64

Shape:
(6, 6)


=== applicants_extra.json ===

First 5 rows:
   id                  skills  education
0   1           [python, sql]  bachelors
1   2  [excel, communication]    masters
2   3                    None  bachelors
3   4            [management]        phd
4 

In [3]:
# Task 2 — Merge Datasets

df = pd.merge(df1, df2, on="id", how="left")

print(df)
print(df.shape)

   id   name   age  experience       city expected_salary  \
0   1    Ali  22.0         1.0  Islamabad           40000   
1   2   Sara   NaN         3.0     Lahore           60000   
2   3  Ahmed  19.0         NaN    Karachi               ?   
3   4  Bilal  30.0         7.0   Peshawar           90000   
4   5   Hira  28.0         5.0        isl           65000   
5   6    NaN  26.0         4.0     Lahore           70000   

                   skills  education  
0           [python, sql]  bachelors  
1  [excel, communication]    masters  
2                    None  bachelors  
3            [management]        phd  
4            [python, ml]       None  
5                     NaN        NaN  
(6, 8)


In [4]:
#Task 3 — Handle Missing Values

import numpy as np

df["age"] = df["age"].fillna(df["age"].median())

df["experience"] = df["experience"].fillna(0)

df["expected_salary"] = df["expected_salary"].replace("?", np.nan)
df["expected_salary"] = df["expected_salary"].astype(float)
df["expected_salary"] = df["expected_salary"].fillna(df["expected_salary"].median())

df["education"] = df["education"].fillna("Unknown")

df["name"] = df["name"].fillna("No_Name")

df["skills"] = df["skills"].apply(lambda x: x if isinstance(x, list) else [])

print(df.isnull().sum())

id                 0
name               0
age                0
experience         0
city               0
expected_salary    0
skills             0
education          0
dtype: int64


In [5]:
#Task 4 — Clean Inconsistent Data

df["city"] = df["city"].replace("isl", "Islamabad")

df["city"] = df["city"].str.lower()

print(df["city"].unique())

['islamabad' 'lahore' 'karachi' 'peshawar']


In [6]:
#Task 5 — Feature Engineering

def experience_level(exp):
    if exp <= 1:
        return "junior"
    elif exp <= 5:
        return "mid"
    else:
        return "senior"

df["experience_level"] = df["experience"].apply(experience_level)

print(df[["experience", "experience_level"]])

   experience experience_level
0         1.0           junior
1         3.0              mid
2         0.0           junior
3         7.0           senior
4         5.0              mid
5         4.0              mid


In [7]:
#Task 6 — Normalize Numeric Features
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols = ["age", "experience", "expected_salary"]

df[cols] = scaler.fit_transform(df[cols])

print(df.head())

   id   name       age  experience       city  expected_salary  \
0   1    Ali  0.272727    0.142857  islamabad              0.0   
1   2   Sara  0.636364    0.428571     lahore              0.4   
2   3  Ahmed  0.000000    0.000000    karachi              0.5   
3   4  Bilal  1.000000    1.000000   peshawar              1.0   
4   5   Hira  0.818182    0.714286  islamabad              0.5   

                   skills  education experience_level  
0           [python, sql]  bachelors           junior  
1  [excel, communication]    masters              mid  
2                      []  bachelors           junior  
3            [management]        phd           senior  
4            [python, ml]    Unknown              mid  


In [10]:
def preprocess_applicants(df):
    df = df.copy()

    df["expected_salary"] = df["expected_salary"].replace("?", np.nan)
    df["expected_salary"] = pd.to_numeric(df["expected_salary"])

    df["age"] = df["age"].fillna(df["age"].median())
    df["experience"] = df["experience"].fillna(0)
    df["expected_salary"] = df["expected_salary"].fillna(df["expected_salary"].median())
    df["education"] = df["education"].fillna("Unknown")
    df["name"] = df["name"].fillna("Unknown")

    df["city"] = df["city"].astype(str).str.lower().replace("isl", "islamabad")

    df["experience_level"] = df["experience"].apply(
        lambda x: "junior" if x <= 1 else ("mid" if x <= 5 else "senior")
    )

    for col in ["age", "experience", "expected_salary"]:
        df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

    return df

In [12]:
#Task 7 — Build a Preprocessing Function

import pandas as pd
import numpy as np

def preprocess_applicants(df):
    df = df.copy()

    df["expected_salary"] = df["expected_salary"].replace("?", np.nan)
    df["expected_salary"] = pd.to_numeric(df["expected_salary"])

    df["age"] = df["age"].fillna(df["age"].median())
    df["experience"] = df["experience"].fillna(0)
    df["expected_salary"] = df["expected_salary"].fillna(df["expected_salary"].median())
    df["education"] = df["education"].fillna("Unknown")
    df["name"] = df["name"].fillna("Unknown")

    df["city"] = df["city"].astype(str).str.lower()
    df["city"] = df["city"].replace("isl", "islamabad")

    def experience_level(x):
        if x <= 1:
            return "junior"
        elif x <= 5:
            return "mid"
        else:
            return "senior"

    df["experience_level"] = df["experience"].apply(experience_level)

    for col in ["age", "experience", "expected_salary"]:
        min_val = df[col].min()
        max_val = df[col].max()
        df[col] = (df[col] - min_val) / (max_val - min_val)

    return df